In [12]:
import pandas as pd
import glob
from time import sleep
from datetime import datetime
import json
from requests import get
from bs4 import BeautifulSoup
from time import sleep
import re

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [176]:
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.common.exceptions import NoSuchElementException, TimeoutException
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.common.keys import Keys

# # NOT FUNCTIONAL automate getting lists from IBKR
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.common.exceptions import NoSuchElementException, TimeoutException

# options = webdriver.ChromeOptions()
# prefs = {"profile.managed_default_content_settings.images": 2}
# options.add_experimental_option("prefs", prefs)

# driver = webdriver.Chrome(options=options)
# driver.maximize_window()

# url = f'https://www.interactivebrokers.com/en/trading/products-exchanges.php#/'
# driver.get(url)
# try:
#     button = driver.find_element(By.ID, 'gdpr-reject-all')
#     WebDriverWait(driver, 5).until(EC.element_to_be_clickable(button))
#     button.click()
# except NoSuchElementException:
#     pass

# time.sleep(3)
# WebDriverWait(driver, 10).until(EC.element_to_be_clickable(By.CSS_SELECTOR, '._btn.text-bold.after-4.text-left'))
# button = driver.find_element(By.CSS_SELECTOR, '._btn.text-bold.after-4.text-left')
# button.click()

# time.sleep(1)
# button = driver.find_element(By.CSS_SELECTOR, '.fa.fa-chevron-left')
# WebDriverWait(driver, 10).until(EC.element_to_be_clickable(button))
# button.click()

# time.sleep(1)
# button = driver.find_element(By.CSS_SELECTOR, '._btn.inversed-black.remove')
# driver.execute_script("window.scrollBy(0,400);")
# time.sleep(2)
# button.click()

# WebDriverWait(driver, 10).until(EC.presence_of_element_located(By.ID, '_dd158'))
# time.sleep(1)
# driver.execute_script("window.scrollBy(0,600);")
# button = driver.find_element(By.ID, '_dd158')
# button.click()

# # while True:
# #     try:
# #         WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".paginacion")))
# #         WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "anuncio")))
# #         html = driver.page_source
# #         soup = BeautifulSoup(html, 'html.parser')
# #         cars += soup.find_all('article', class_='anuncio')
        
# #         pages = driver.find_element(By.CLASS_NAME, "paginacion")
# #         next_button = pages.find_elements(By.TAG_NAME, "li")[-1]
# #         next_url = next_button.find_element(By.TAG_NAME, "a").get_attribute("href")
# #         driver.get(next_url)

# #     except NoSuchElementException:
# #         print('no such element')
# #         print(next_url)
# #         break
# #     except TimeoutException:
# #         print('timeout')
# #         print(next_url)
# #         break

# # driver.quit()

In [172]:
# Load ETF csvs
path = "data/etfs-IBKR/*.csv" 
all_files = glob.glob(path)

df_list = []
for filename in all_files:
    df = pd.read_csv(filename, index_col=None, header=0, encoding='utf-8')
    df_list.append(df)

df = pd.concat(df_list, axis=0, ignore_index=True)

df.columns = df.columns.str.lower()
df.description = df.description.replace('&amp;', '&', regex=True)

# Elaborate region values
country_dict = {
    'TW': 'TW - Taiwan',
    'US': 'US - United States',
    'DE': 'DE - Germany',
    'LU': 'LU - Luxembourg',
    'JP': 'JP - Japan',
    'SG': 'SG - Singapore',
    'JE': 'JE - Jersey',
    'FR': 'FR - France',
    'IE': 'IE - Ireland',
    'AU': 'AU - Australia',
    'CH': 'CH - Switzerland',
    'HK': 'HK - Hong Kong',
    'CA': 'CA - Canada',
    'MX': 'MX - Mexico',
    'IN': 'IN - India',
    'IL': 'IL - Israel',
    'LI': 'LI - Liechtenstein',
    'NL': 'NL - Netherlands',
    'NO': 'NO - Norway',
    'KY': 'KY - Cayman Islands',
    'SE': 'SE - Sweden',
    'ES': 'ES - Spain',
    'GG': 'GG - Guernsey',
    'PL': 'PL - Poland',
    'HU': 'HU - Hungary',
    'XX': 'XX - Other',
    'ZA': 'ZA - South Africa',
    'CN': 'CN - China'
}
df['region'] = df['region'].map(country_dict)

# Filter to EUR etfs
eur = df[df['currency'] == 'EUR']
eur = eur.drop(['currency', 'product', 'symbol'], axis=1)

In [ ]:
# Get ISIN from ibkr
from ib_async import *
import pandas as pd
util.startLoop()

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=1)

etfs = []
ib.sleep(3)
for etf_symbol in eur['ibkr-symbol']:
    etf_details = ib.reqContractDetails(Stock(etf_symbol, 'SMART', 'EUR'))
    etfs.append([etf_symbol, etf_details[0]])
    
etfs = [etf for etf in etfs if etf[1] != []]

ib.disconnect()

In [ ]:
# Add isin column, and url columns
dict_from_list = {item[0]: item[1].secIdList[0].value for item in etfs}
eur['isin'] = eur['ibkr-symbol'].map(dict_from_list)
eur = eur.dropna()

def getJustEtfsURL(isin):
    return f'https://www.justetf.com/es/etf-profile.html?isin={isin}'

def getExtraEtfsURL(isin):
    return f'https://extraetf.com/es/etf-profile/{isin}'

eur['just-etf'] = eur['isin'].apply(getJustEtfsURL)
eur['extra-etf'] = eur['isin'].apply(getExtraEtfsURL)

eur.to_csv('data/eur.csv', index=False)

In [17]:
# JustETF scraping functions
def get_basics(soup):
    try:
        table = soup.find('table', class_='table etf-data-table')

        basics = {}
        for row in table.find_all('tr'):
            label_elem = row.find('td', class_='vallabel')
            if label_elem:
                label = label_elem.text.strip()
                label = [text.lower().strip() for text in label.split(' ')]
                label = '_'.join(label)

                value_elem = label_elem.find_next_sibling()
                if value_elem:
                    value = value_elem.get_text(strip=True)
                    basics[label] = value
                
            else: 
                print('no label found')
        
        return basics
    
    except (AttributeError, ValueError, IndexError) as e:
        basics = {'índice': None,
                'foco_de_la_inversión': None,
                'patrimonio_del_fondo': None,
                'gastos_corrientes': None,
                'replicación': None,
                'estructura_jurídica': None,
                'riesgo_estratégico': None,
                'sostenibilidad': None,
                'divisa_del_fondo': None,
                'riesgo_de_divisa': None,
                'volatilidad_1_año_(in_eur)': None,
                'fecha_de_inicio/_de_cotización': None,
                'política_de_distribución': None,
                'frecuencia_de_distribución': None,
                'domicilio_del_fondo': None,
                'proveedor_de_fondo': None,}
        
        return basics


def get_intro_data(soup):
    try:
        intro_elems = soup.find_all('div', class_='d-flex d-flex-column')

        intro = {}
        for elem in intro_elems:
            label = elem.find('div', class_='vallabel').text.strip().lower()
            value = elem.find('div', class_='val bold').text.strip()
            intro[label.replace(' ','_')] = value

        return intro
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'INTRO = {e}')

        return {'ter': None,
                'política_de_distribución': None,
                'replicación': None,
                'tamaño_del_fondo': None,
                'participaciones': None,}


def get_current_dividend(soup):
    try:
        table = soup.find('table', class_='table m-0')
        
        current_dividend = {}
        for row in table.find_all('tr'):
            label_elem = row.find('td', class_='vallabel')
            if label_elem:
                label = label_elem.text.strip()
                value_elem = label_elem.find_next_sibling()
                if value_elem:
                    value = value_elem.get_text(strip=True)
                    current_dividend[label.lower().replace(' ','_')] = value

        return current_dividend
    
    except (AttributeError, ValueError, IndexError) as e:
        # print(f'    CURRENT_DIVIDENDS = {e}')
        
        return {'rentabilidad_actual_de_los_dividendos': None, 'dividendos_(últimos_12_meses)': None},


def get_historic_dividend(soup):
    try:
        table = soup.find('table', class_='table m-0 mobile-table')
        
        historic_dividend = []
        for row in table.find_all('tr'):
            cols = row.find_all('td')
            if cols:
                period = cols[0].text.strip()
                dividend = cols[1].text.strip()
                yield_percent = cols[2].text.strip()
                historic_dividend.append((period, dividend, yield_percent))

        return {'historic_dividend': historic_dividend}

    except (AttributeError, ValueError, IndexError) as e:
        # print(f'    HISTORIC_DIVIDENDS = {e}')

        return {'historic_dividend':[]}


def get_exchanges(soup):
    try:
        table = soup.find('table', class_='table mobile-table')

        exchanges = []
        for row in table.find_all('tr'):
            cols = row.find_all('td')
            if cols:
                period = cols[0].text.strip()
                dividend = cols[1].text.strip()
                yield_percent = cols[2].text.strip()
                bloomberg_inav = cols[3].text.strip()
                reuters_inav = cols[4].text.strip()
                market_maker = cols[5].text.strip()
                exchanges.append((period, dividend, yield_percent, bloomberg_inav, reuters_inav, market_maker))

        return {'exchanges': exchanges}
        
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    EXCHANGES = {e}')

        return {'exchanges': []}



def get_error(soup):
    try:
        error_elem = soup.find('span', class_='feedbackPanelERROR')
        if 'fund has been liquidated or merged' in error_elem.text:
            return {'liquidated': True}
        else:
            return {'liquidated': False}
        
    except AttributeError as e:
        # print(f'    ERRORS = {e}')

        return {'liquidated': False}


# def get_pdfs(soup):
#     try:
#         heading_elems = soup.find_all('h3')

#         for heading in heading_elems:
#             if heading.text.strip() == 'Documentos':
#                 table = heading.parent.find('table', class_='table mb-0')
#                 pdf_elem = table.find_all('a', class_='download-link')

#                 pdfs = {}
#                 for pdf in pdf_elem:
#                     pdf_text = re.sub(r'\s+\(', '_', pdf.text.strip())
#                     pdf_text = re.sub(r'\)', '', pdf_text).lower()
#                     pdfs[pdf_text] = pdf['href']
                    
#                 return pdfs
            
#         return {'factsheet_en': None,
#                 'kid_de': None,
#                 'prospectus_en': None,
#                 'semi-annual report_en': None,
#                 'annual report_en': None}
    
#     except (AttributeError, ValueError, IndexError) as e:
#         print(f'    PDFS = {e} 10')
    
#         return {'factsheet_en': None,
#                 'kid_de': None,
#                 'prospectus_en': None,
#                 'semi-annual report_en': None,
#                 'annual report_en': None}
    

def get_indexNcategories(soup):
    try:
        heading_elems = soup.find_all('div', class_='h4')

        for heading in heading_elems:
            if heading.text.strip() == 'Similares ETF a través de la búsqueda ETF':
                table = heading.parent.find('table', class_='table mb-0')

                indexNcategories = {}
                for row in table.find_all('tr'):
                    text = row.text.strip().split(':')
                    indexNcategories[text[0].lower()] = text[1].strip()
                    
                return indexNcategories
        return {'index': None, 'category': None}
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    INDEX+CATEGORIES = {e}')
        return {'index': None, 'categories': None}


def get_top_10(soup):
    try:
        # Get top 10 elem + num holdings
        top_10_elem = soup.find('div', class_='constituents-top mb-3 clearfix') # weight top 10
        num_holdings = top_10_elem.find('div', class_='left').find_all('div')[1]
        num_holdings = num_holdings.get_text().split(' ')[-1]
        num_holdings = int(num_holdings.replace(',','').replace('.',''))
        
        top_10_percentage = top_10_elem.find('div', class_='right').find('span')
        top_10_percentage = top_10_percentage.get_text().strip('%')
        top_10_percentage = float(top_10_percentage.replace(',','')) / 100

        return {'top_10_holdings': top_10_percentage, 'num_holdings': num_holdings}
    
    except (AttributeError, ValueError, IndexError) as e:
        # print(f'    TOP 10 HOLDINGS = {e} 10')
        
        return {'top_10_holdings': None, 'num_holdings': None}


def get_holdings(soup):
    try:
        heading_elems = soup.find_all('h3')
        
        for heading in heading_elems:
            if heading.text.strip() == 'Las 10 mayores participaciones':
                table = heading.parent.find('table', class_='table mb-0')

                holdings = []
                for row in table.find_all('tr'):
                    code_elem = row.find('span')
                    percentage_elem = row.find('div', class_='right ws').find('span')
                    if code_elem and percentage_elem:
                        code = code_elem.text.strip()
                        percentage = percentage_elem.text.strip()
                        holdings.append((code, percentage))

                return {'holdings': holdings}
        return {'holdings': []}
        
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    HOLDINGS = {e}')

        return {'holdings': []}


def get_countries(soup):
    try:
        heading_elems = soup.find_all('h3')
        
        for heading in heading_elems:
            if heading.text.strip() == 'Países':
                table = heading.parent.find('table', class_='table mb-0')

                countries = []
                for row in table.find_all('tr'):
                    code_elem = row.find('td')
                    percentage_elem = row.find('div', class_='right').find('span')
                    if code_elem and percentage_elem:
                        code = code_elem.text.strip()
                        percentage = percentage_elem.text.strip()
                        countries.append((code, percentage))

                return {'countries': countries}
        return {'countries': []}
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    COUNTRIES = {e}')
        
        return{'countries': []}
        

def get_sectors(soup):
    try:
        heading_elems = soup.find_all('h3')

        for heading in heading_elems:
            if heading.text.strip() == 'Sectores':
                table = heading.parent.find('table', class_='table mb-0')

                sectors = []
                for row in table.find_all('tr'):
                    code_elem = row.find('td')
                    percentage_elem = row.find('div', class_='right').find('span')
                    if code_elem and percentage_elem:
                        code = code_elem.text.strip()
                        percentage = percentage_elem.text.strip()
                        sectors.append((code, percentage))

                return {'sectors': sectors}
        return {'sectors': []}
    
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    SECTORS = {e}')

        return {'sectors': []}

In [186]:
# Scrape JustETF

# url_data = []

# check existing data to aviod redundant requests
if url_data:
    url_set = set([sublist[0] for sublist in url_data])
    remaining_urls = eur['just-etf'].to_list()
    remaining_urls = [x for x in remaining_urls if x not in url_set]
else:
    url_data = []
    remaining_urls = eur['just-etf'].to_list()

counter = len(url_data)
for url in remaining_urls:
    print(f'{counter} == {url}')
    counter += 1

    response = get(url)
    if response.status_code != 200:
        print('got bad response !=200')
        break
    soup = BeautifulSoup(response.text, 'html.parser')

    basics = get_basics(soup)
    # pdfs = get_pdfs(soup)
    indexNcategories = get_indexNcategories(soup)
    top10 = get_top_10(soup)
    holdings = get_holdings(soup)
    countries = get_countries(soup)
    sectors = get_sectors(soup)
    current_dividend = get_current_dividend(soup)
    historic_dividends = get_historic_dividend(soup)
    exchanges = get_exchanges(soup)
    intro_data = get_intro_data(soup)        
    error = get_error(soup)

    url_data.append([url, exchanges, historic_dividends, current_dividend, holdings, sectors, countries, top10, basics, indexNcategories, intro_data, error])

    sleep(0.5)

with open("/data/ibkr_justetfs_scraped.json", "w") as file:
    # Use json.dump to write the list to a file
    json.dump(url_data, file)

0 == https://www.justetf.com/en/etf-profile.html?isin=LU0671493277
1 == https://www.justetf.com/en/etf-profile.html?isin=IE0004J37T45
2 == https://www.justetf.com/en/etf-profile.html?isin=FR0010756114
3 == https://www.justetf.com/en/etf-profile.html?isin=LU0879397742
4 == https://www.justetf.com/en/etf-profile.html?isin=LU0879399441


In [176]:
# ExtraETF scraping functions
def getTopInfo(soup):
    try:
        container = soup.find('div', class_="top-info desktop-only")
        info = container.find_all('div')
        top_data = {}
        for i in info:
            label = i.find_all('span')[1].text.strip().lower()
            label = [text.strip() for text in label.split(' ')]
            label = '_'.join(label)
            
            value = i.find_all('span')[0].text.strip().lower()
            try: 
                value = int(value)
            except ValueError:
                value = value.replace('\xa0', '')
                if '%' in value:
                    value = value.replace('%', '')
                    value = value.replace(',', '.')
                    try:
                        value = float(value)
                    except ValueError:
                        pass
                elif 'm€' in value:
                    if '<' in value:
                        value = value.replace('<', '')
                        value = value.replace('mil', '')
                        value = value.replace('m€', '000000')
                        try:
                            value = int(value.replace(',', '').strip())
                        except ValueError:
                            pass
                    else: 
                        value = value.replace('mil', '')
                        value = value.replace('m€', '0000')
                        try:
                            value = int(value.replace(',', '').strip())
                        except ValueError:
                            pass
                
            top_data[label] = value
        return top_data
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    TOP_INFO = {e}')
        return {}


def getBasics(soup):
    try:
        heading_elems = soup.find_all('h3')
        for heading in heading_elems:
            if heading.text.strip() == 'Datos básicos':
                table = heading.parent.parent.find('table', class_='table last-column-end')

                basics = []
                for row in table.find_all('tr'):
                    elems = row.find_all('td')
                    label = elems[0].text.strip().lower()
                    label = [text.strip() for text in label.split(' ')]
                    label = '_'.join(label)

                    value = elems[1].text.strip().lower()
                    try: 
                        value = int(value)
                    except ValueError:
                        value = value.replace('\xa0', '')
                        if '%' in value:
                            value = value.replace('%', '')
                            value = value.replace(',', '.')
                            value = float(value)
                        elif 'm€' in value:
                            if '<' in value:
                                value = value.replace('<', '')
                                value = value.replace('mil', '')
                                value = value.replace('m€', '000000')
                                try:
                                    value = int(value.replace(',', '').strip())
                                except ValueError:
                                    pass
                            else: 
                                value = value.replace('mil', '')
                                value = value.replace('m€', '0000')
                                try:
                                    value = int(value.replace(',', '').strip())
                                except ValueError:
                                    pass
                    basics.append((label, value))
                
                basics = {i[0]:i[1] for i in basics}
                return basics
        return {}
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    BASICS = {e}')
        return {}

def getIndex(soup):
    try:
        heading_elems = soup.find_all('h3')
        for heading in heading_elems:
            if 'Información sobre' in heading.text.strip():
                table = heading.parent.parent.find('table', class_='table last-column-end')

                index = []
                for row in table.find_all('tr'):
                    elems = row.find_all('td')
                    label = elems[0].text.strip().lower()
                    label = [text.strip() for text in label.split(' ')]
                    label = '_'.join(label)

                    value = elems[1].text.strip().lower()
                    try: 
                        value = int(value)
                    except ValueError:
                        value = value.replace('\xa0', '')
                        if '%' in value:
                            try:
                                value = value.replace('%', '')
                                value = value.replace(',', '.')
                                value = float(value)
                            except ValueError:
                                pass
                        elif 'm€' in value:
                            value = value.replace('mil', '')
                            value = value.replace('m€', '0000')
                            try:
                                value = int(value.replace(',', ''))
                            except ValueError:
                                pass
                    index.append((label, value))
                
                index = {i[0]:i[1] for i in index}
                return index
        print('    INDEX_DATA h3 not found')
        return {}
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    INDEX_DATA = {e}')
        return {}

def getDividends(soup):
    try:
        heading_elems = soup.find_all('h3')
        for heading in heading_elems:
            if 'Distribuciones' in heading.text.strip():
                table = heading.parent.parent.find('table', class_='table last-column-end')

                dividends = []
                table = table.find('tbody')
                for row in table.find_all('tr'):
                    elems = row.find_all('td')
                    label = elems[0].text.strip().lower()
                    label = [text.strip() for text in label.split(' ')]
                    label = label[0]

                    value = elems[1].text.strip().lower()
                    value = value.replace('<', '.')
                    value = value.replace(',', '.')
                    value = value.replace('€', '')
                    try:
                        value = float(value)
                    except ValueError:
                        pass
                    percentage = elems[2].text.strip().lower()
                    percentage = percentage.replace('\xa0%', '')
                    try:
                        percentage = float(percentage.replace(',', '.'))/100
                    except ValueError:
                        pass
                    dividends.append(((label + '_dividend_value'), value))
                    dividends.append(((label + '_dividend_percentage_of_distribution'), percentage))
                
                dividends = {i[0]:i[1] for i in dividends}
                return dividends
        print(f'    DIVIDIEND h3 not found')
        return {}
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    DIVIDIENDS = {e}')
        return {}

def getPositions(soup): # add components to url
    try:
        heading_elems = soup.find_all('h3')
        for heading in heading_elems:
            if 'Top' in heading.text.strip():
                table = heading.parent.parent.find('table')

                col_pos = []
                positions = []
                body = table.find('tbody')
                for row in body.find_all('tr'):
                    elems = row.find_all('td')
                    label = elems[2].text.strip().lower()

                    value = elems[-1].text.strip().lower()
                    value = value.replace(',', '.')
                    value = value.replace('\xa0%', '')
                    try:
                        value = float(value)/100
                    except ValueError:
                        pass

                    positions.append((label, value))
                positions = {i[0]:i[1] for i in positions}
                col_pos.append(('positions', positions))
                
                foot = table.find('tfoot')
                value = foot.text.replace('Suma:\xa0', '')
                value = value.replace(',', '.')
                value = value.replace('\xa0%', '')
                try:
                    value = float(value)/100
                except ValueError:
                    pass
                col_pos.append(('top_10_positions', value))
                col_pos = {i[0]:i[1] for i in col_pos}

                return col_pos
            
        print(f'    POSITIONS h3 not found')
        return {}
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    POSITIONS = {e}')
        return {}

def getCountries(soup): # add components to url
    try:
        heading_elems = soup.find_all('h3')
        for heading in heading_elems:
            if 'países' in heading.text.strip():
                table = heading.parent.parent.find_all('div', class_='item ng-star-inserted')
                index = []
                for row in table:
                    label = row.find('div', class_='progress-block').text.strip().lower()
                    label = [text.strip() for text in label.split(' ')]
                    label = '_'.join(label)

                    value = row.find('div', class_='value-block').text.strip().lower()
                    value = value.replace('\xa0', '')
                    if '%' in value:
                        value = value.replace('%', '')
                        value = value.replace(',', '.')
                        try:
                            value = float(value)/100
                        except ValueError:
                            pass
                    index.append((label, value))
                
                index = {i[0]:i[1] for i in index}
                return index
        print('    COUNTRIES h3 not found')
        return {}
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    COUNTRIES = {e}')
        return {}

def getSectors(soup): # add components to url
    try:
        heading_elems = soup.find_all('h3')
        for heading in heading_elems:
            if 'Sectores industriales' in heading.text.strip():
                table = heading.parent.parent.find_all('div', class_='item ng-star-inserted')
                index = []
                for row in table:
                    label = row.find('div', class_='progress-block').text.strip().lower()
                    label = [text.strip() for text in label.split(' ')]
                    label = '_'.join(label)

                    value = row.find('div', class_='value-block').text.strip().lower()
                    value = value.replace('\xa0', '')
                    if '%' in value:
                        value = value.replace('%', '')
                        value = value.replace(',', '.')
                        try:
                            value = float(value)/100
                        except ValueError:
                            pass
                    index.append((label, value))
                
                index = {i[0]:i[1] for i in index}
                return index
        print('    SECTORS h3 not found')
        return {}
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    SECTORS = {e}')
        return {}
    
def getStars(soup):
    try:
        container = soup.find('div', class_='star-container')
        stars = container.parent['class']
        return stars
    except (AttributeError, ValueError, IndexError) as e:
        print(f'    STARS = {e}')
        return {}

In [193]:
# Scrape ExtraETF

# url_data = []

# check existing data to aviod redundant requests
if url_data:
    url_set = set([sublist[0] for sublist in url_data])
    remaining_urls = eur['extra-etf'].to_list()
    remaining_urls = [x for x in remaining_urls if x not in url_set]
else:
    url_data = []
    remaining_urls = eur['extra-etf'].to_list()
   

counter = len(url_data)
twenties = 0
for url in remaining_urls:
    print(f'{counter} == {url}')
    counter += 1

    response = get(url)
    if response.status_code != 200:
        print(f'got response {response.status_code}')
        if response.status_code == 404:
            continue
        else:
            break
    soup = BeautifulSoup(response.text, 'html.parser')

    stars = getStars(soup)
    top_info = getTopInfo(soup)
    basics = getBasics(soup)
    index = getIndex(soup)
    dividends = getDividends(soup)

    # sleep(1.5)
    
    response = get(url + '?tab=components')
    if response.status_code != 200:
        print(f'got response {response.status_code}')
        if response.status_code == 404:
            continue
        else:
            break
    soup = BeautifulSoup(response.text, 'html.parser')

    positions = getPositions(soup)
    countries = getCountries(soup)
    sectors = getSectors(soup)

    url_data.append([url, stars, top_info, basics, index, dividends, positions, countries, sectors])

    # sleep(1.5)

with open("data/ibkr_extraetfs_scraped.json", "w") as file:
    # Use json.dump to write the list to a file
    json.dump(url_data, file)


1094 == https://extraetf.com/es/etf-profile/IE00BKS8QM96
got response 404
1095 == https://extraetf.com/es/etf-profile/IE00BKS8QN04
got response 404
1096 == https://extraetf.com/es/etf-profile/JE00BYQY3Z98
got response 404
1097 == https://extraetf.com/es/etf-profile/IE00B878KX55
got response 404
1098 == https://extraetf.com/es/etf-profile/IE00B8GKPP93
got response 404
1099 == https://extraetf.com/es/etf-profile/IE00B8JF9153
got response 404
1100 == https://extraetf.com/es/etf-profile/IE00B8HGT870
got response 404
1101 == https://extraetf.com/es/etf-profile/IE00B6X4BP29
got response 404
1102 == https://extraetf.com/es/etf-profile/IE00B8JVMZ80
got response 404
1103 == https://extraetf.com/es/etf-profile/IE00B8KD3F05
got response 404
1104 == https://extraetf.com/es/etf-profile/XS2842095320
got response 404
1105 == https://extraetf.com/es/etf-profile/LU1834984798
got response 404
1106 == https://extraetf.com/es/etf-profile/JE00B3RQ6R05
got response 404
1107 == https://extraetf.com/es/etf-pr